# 생성형 AI 윤리 가이드북 RAG

## 환경 설정

In [19]:
import json
from dotenv import load_dotenv

load_dotenv()

PDF_PATH = "data/생성형_AI윤리_가이드북_RAG시험용_30쪽.pdf"
QUESTION_PATH = "data/student_13_questions.json"

In [20]:
f = open(QUESTION_PATH, encoding="utf-8")
data = json.load(f)
f.close()

questions = data["questions"]
print(len(questions))

for q in questions:
    print(q["question"])

3
대학교에서 생성형 AI 활용 시 표절과 부정행위 문제를 어떻게 판단하고 대응하는지, 관련 가이드라인과 사례를 근거로 비교·분석하세요.
생성형 AI를 활용할 때 개인정보 유출 위험을 줄이기 위해 어떤 점을 주의해야 하는지 사례를 들어 설명하세요.
생성형 AI를 활용해 업무 자료를 만들 때 주의해야 할 세 가지 사항은 무엇인가요?


## 1단계. 문서 로드

In [21]:
from langchain_community.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader(PDF_PATH)
pages = loader.load()
print(len(pages))

30


In [22]:
for d in pages:
    print(d.metadata["page"] + 1, len(d.page_content.strip()))

1 0
2 1301
3 1614
4 822
5 1399
6 1130
7 1106
8 1136
9 1128
10 1266
11 918
12 1200
13 1236
14 1262
15 1151
16 928
17 784
18 783
19 496
20 1161
21 950
22 1024
23 539
24 1128
25 1292
26 1441
27 1709
28 1267
29 814
30 223


In [23]:
pages = pages[1:]
print(len(pages))
print(pages[0].page_content[:300])

29
08
생성형 AI윤리 가이드북
PART
생성형 AI란?
01
생성형 AI가 무엇일까요?
생성형 AI(Generative AI)란, 대규모 데이터와 패턴을 학습하고 기존의 데이터를 활용하여 이용자의 요구에 
따라 텍스트, 이미지, 비디오, 음악, 코딩 등 새로운 결과를 만들어 내는 인공지능 기술입니다. 생성형 AI는 
누구나 쉽게 활용할 수 있도록 설계되었고, 굉장히 다양한 영역에서 유용하게 활용될 수 있기 때문에 인공지능 
분야의 새로운 화두가 되었습니다. 
지난 2016년에 구글 딥마인드(Google DeepMind)가 개발한 인


## 2단계. 청킹

In [24]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)

chunks = splitter.split_documents(pages)
print(len(chunks))

77


## 3단계. 임베딩

In [25]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

## 4단계. 벡터스토어

In [26]:
from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
)
print(vector_store._collection.count())

154


In [27]:
result = vector_store.similarity_search(questions[1]["question"], k=3)
print(result[0].page_content)

생성형 AI를 활용해 업무 자료를 만들 때는 아래 3가지 사항을 주의해야 합니다.
➊ 자료의 진위 여부(환각 현상, Hallucination)
생성형 AI는 거짓 정보를 사실인 것처럼 제공하는 환각 현상이 있으므로, 업무 자료로 추천해 준 
통계나 데이터가 거짓이거나 제시된 내용에 오류가 있을 수 있습니다. 때문에 정보의 진위 여부에 
대해서는 최종적으로 사실 확인(팩트 체크)을 꼭 해야 해요. 
➋ 회사 기밀 유출 위험
생성형 AI에 입력되는 내용들은 외부 서버에 저장되고 AI의 학습에 재이용되기도 하는 등 외부 유출의 
위험이 있습니다. 따라서 회사에서 업무 자료를 만들 때 회사 기밀이나 개인정보 등 민감한 정보는 
생성형 AI에 입력하지 말아야 해요.
➌ 업무 자료의 책임 소재
생성형 AI로 만든 자료를 업무 문서로 그대로 제출 및 활용하는 것은 책임 소재 측면에서 바람직하지 
않습니다. 생성형 AI는 업무 자료 작성에 보조적으로 활용하고, 최종 자료의 작성과 완성은 작성자


## 5단계. 리트리버

In [28]:
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever

bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 10

chroma_retriever = vector_store.as_retriever(search_kwargs={"k": 10})

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, chroma_retriever],
    weights=[0.5, 0.5],
)

In [29]:
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain.retrievers import ContextualCompressionRetriever

model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3")
compressor = CrossEncoderReranker(model=model, top_n=4)

retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=ensemble_retriever,
)

d:\psj0902\test2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 393/393 [00:01<00:00, 375.53it/s]


## 6단계. 프롬프트

In [30]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
당신은 생성형 AI 윤리 가이드북을 바탕으로 답변하는 도우미입니다.
아래 [문서] 내용만 근거로 질문에 답하세요. 문서에 없는 내용은 추측해서 쓰지 마세요.
질문에 직접 답하는 문장으로 시작하고, 질문과 관계없는 내용은 빼세요.
질문이 요구한 항목(예: 세 가지, 비교, 사례)만 빠짐없이 3~5문장으로 답하세요.

[문서]
{context}

[질문]
{question}

[답변]
""")

## 7단계. LLM

In [31]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o", temperature=0)

## 8단계. 출력 파서

In [32]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

## 9단계. 체인

In [33]:
def format_docs(docs):
    text = ""
    for d in docs:
        text += d.page_content + "\n\n"
    return text

In [34]:
from langchain_core.runnables import RunnablePassthrough

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | parser
)

In [35]:
answer = chain.invoke(questions[0]["question"])
print(answer)

대학교에서 생성형 AI 활용 시 표절과 부정행위 문제를 판단하고 대응하는 방식은 각 대학의 가이드라인과 사례를 통해 확인할 수 있습니다. 연세대의 경우, 챗GPT를 활용한 대필이 의심되는 과제에 대해 표절 정황이 명백하다고 판단하여 0점 처리하는 방식으로 대응했습니다. 중앙대는 학생들에게 온라인 표절 관련 교육을 실시하고, "챗GPT를 활용해 표절하지 않겠다"는 서약서를 받는 방식으로 예방 조치를 취했습니다. 이러한 사례들은 생성형 AI를 활용한 결과물을 그대로 제출하는 것이 학문적 진실성을 위반할 수 있음을 보여주며, 각 대학은 이를 방지하기 위해 가이드라인을 제정하고 학생들에게 이를 준수하도록 요구하고 있습니다.
